In [2]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import spacy
import pickle
import random

import os
for dirname, _, filenames in os.walk('data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

data\Alice Clark CV.pdf
data\AshlyLauResume.pdf
data\Sample_Resume.pdf
data\Smith Resume.pdf
data\train_data.pkl
data\train_data.txt
data\Xinni_Chng.pdf


In [3]:
train_data = pickle.load(open('data/train_data.pkl','rb'))
train_data[0]

('Govardhana K Senior Software Engineer  Bengaluru, Karnataka, Karnataka - Email me on Indeed: indeed.com/r/Govardhana-K/ b2de315d95905b68  Total IT experience 5 Years 6 Months Cloud Lending Solutions INC 4 Month • Salesforce Developer Oracle 5 Years 2 Month • Core Java Developer Languages Core Java, Go Lang Oracle PL-SQL programming, Sales Force Developer with APEX.  Designations & Promotions  Willing to relocate: Anywhere  WORK EXPERIENCE  Senior Software Engineer  Cloud Lending Solutions -  Bangalore, Karnataka -  January 2018 to Present  Present  Senior Consultant  Oracle -  Bangalore, Karnataka -  November 2016 to December 2017  Staff Consultant  Oracle -  Bangalore, Karnataka -  January 2014 to October 2016  Associate Consultant  Oracle -  Bangalore, Karnataka -  November 2012 to December 2013  EDUCATION  B.E in Computer Science Engineering  Adithya Institute of Technology -  Tamil Nadu  September 2008 to June 2012  https://www.indeed.com/r/Govardhana-K/b2de315d95905b68?isid=rex-

In [4]:
# Load Blank Model
nlp = spacy.blank('en')

def train_model(train_data):
    # Remove all pipelines and add NER pipeline from the model
    if 'ner'not in nlp.pipe_names:
        ner = nlp.create_pipe('ner')
        # adding NER pipeline to nlp model
        nlp.add_pipe(ner,last=True)
    
    #Add labels in the NLP pipeline
    for _, annotation in train_data:
        for ent in annotation.get('entities'):
            ner.add_label(ent[2])
    
    #Remove other pipelines if they are there
    other_pipes = [pipe for pipe in nlp.pipe_names if pipe != 'ner']
    with nlp.disable_pipes(*other_pipes):  # only train NER
        optimizer = nlp.begin_training()
        for itn in range(10): # train for 10 iterations
            print("Starting iteration " + str(itn))
            random.shuffle(train_data)
            losses = {}
            index = 0
            for text, annotations in train_data:
                try:
                    nlp.update(
                        [text],  # batch of texts
                        [annotations],  # batch of annotations
                        drop=0.2,  # dropout - make it harder to memorise data
                        sgd=optimizer,  # callable to update weights
                        losses=losses)
                except Exception as e:
                    pass
                
            print(losses)
# Start Training model
train_model(train_data)

C:\Users\drnwe\anaconda3\envs\galaxie\Lib\site-packages\spacy\language.py:635: UserWarning: [W033] Training a new parser or NER using a model with an empty lexeme normalization table. This may degrade the performance to some degree. If this is intentional or this language doesn't have a normalization table, please ignore this warning.
  proc.begin_training(
C:\Users\drnwe\anaconda3\envs\galaxie\Lib\site-packages\spacy\language.py:635: UserWarning: [W034] Please install the package spacy-lookups-data in order to include the default lexeme normalization table for the language 'en'.
  proc.begin_training(
C:\Users\drnwe\anaconda3\envs\galaxie\Lib\site-packages\spacy\language.py:482: UserWarning: [W030] Some entities could not be aligned in the text "Nidhi Pandit Test Engineer - Infosys Limited  - Em..." with entities "[(3132, 3611, 'Skills'), (3005, 3083, 'Skills'), (...". Use `spacy.gold.biluo_tags_from_offsets(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities (w

Starting iteration 0


C:\Users\drnwe\anaconda3\envs\galaxie\Lib\site-packages\spacy\language.py:482: UserWarning: [W030] Some entities could not be aligned in the text "Shaheen Unissa SAP ABAP Consultant  Hyderabad, Tel..." with entities "[(12088, 12384, 'Skills'), (12044, 12048, 'Graduat...". Use `spacy.gold.biluo_tags_from_offsets(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities (with BILUO tag '-') will be ignored during training.
  gold = GoldParse(doc, **gold)
C:\Users\drnwe\anaconda3\envs\galaxie\Lib\site-packages\spacy\language.py:482: UserWarning: [W030] Some entities could not be aligned in the text "Avani Priya - Email me on Indeed: indeed.com/r/Ava..." with entities "[(368, 409, 'Email Address'), (314, 334, 'College ...". Use `spacy.gold.biluo_tags_from_offsets(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities (with BILUO tag '-') will be ignored during training.
  gold = GoldParse(doc, **gold)
C:\Users\drnwe\anaconda3\envs\galaxie\Lib\site-packa

{'ner': 14333.081825699883}
Starting iteration 1
{'ner': 9971.121412031216}
Starting iteration 2
{'ner': 11143.18710184222}
Starting iteration 3
{'ner': 6685.601913822221}
Starting iteration 4
{'ner': 5522.478050267983}
Starting iteration 5
{'ner': 5554.457634209462}
Starting iteration 6
{'ner': 5179.291173582469}
Starting iteration 7
{'ner': 5710.729607669039}
Starting iteration 8
{'ner': 4933.888697205789}
Starting iteration 9
{'ner': 4514.988701782894}


In [5]:
nlp.to_disk('nlp_ner_model')

In [6]:
nlp_model = spacy.load('nlp_ner_model')

In [7]:
doc = nlp_model(train_data[0][0])
for ent in doc.ents:
    print(f"{ent.label_.upper():{30}}-{ent.text}")

NAME                          -Pawan Nag
DESIGNATION                   -Microsoft
DESIGNATION                   -System Engineer
LOCATION                      -Delhi
EMAIL ADDRESS                 -indeed.com/r/Pawan-Nag/e14493f28cb72022
COMPANIES WORKED AT           -Microsoft
DESIGNATION                   -System Engineer
DEGREE                        -AMIE sec A
SKILLS                        -Skill Summary: Operating Systems: • Windows Server 2000, 2003, 2008, 2012, 2012R2. • Exchange 2003, 2007, 2010, 2013,  Date […] (Name) Pawan Nag


In [35]:
import sys, fitz
fname = 'data/Xinni_Chng.pdf'
doc = fitz.open(fname)
all_text=""

for page_number in range(doc.page_count):
        # Select the current page
        page = doc[page_number]

        # Extract text from the page
        text = page.get_text()

        # Print or process the extracted text
        #print(f"Page {page_number + 1}:\n{text}\n")
        all_text+=" ".join(text.split('\n'))
print(all_text)

SF Bay Area (669) 262-8183 hello@xinni.co    http://xinni.co USER EXPERIENCE DESIGNER xinni chng S K I L L S DeSign Responsive Web Design,  Mobile design, Visual  Design, game Design,  illustration MethoDS User Research, Competitive  Analysis, Wireframing,  Prototyping, Usability  testing, Statistical Analysis toolS Adobe Creative Suite,  Sketch, Principle, invision,  Balsamiq, R Studio, Cogtool teChnologieS htMl, CSS, JavaScript,  C++, Java, JQuery, Angular,  React, ionic, grunt, gulp,  Qt, SQl, Firebase E X P E R I E N C E Google — User experience Design intern MAY 2016 - PRESENT Bridging the world - collaborating with designers, engineers and product  managers to build an improved experience for google translate. NUS School of Computing — teaching Assistant DEC 2015 - MAY 2016 Collaborated with the lecturer to craft a new syllabus for User interface  Development; Conducted lectures on responsive design and best web  practices to 50+ students and held one-on-one help sessions. IDA

In [36]:
doc = nlp_model(all_text)
for ent in doc.ents:
    print(f'{ent.label_.upper():{30}}- {ent.text}')

NAME                          - SF Bay
DESIGNATION                   - Area (669) 262-8183 hello@xinni.co    http://xinni.co USER EXPERIENCE DESIGNER xinni chng S K I L L S DeSign Responsive Web Design,  Mobile design, Visual  Design, game Design,
LOCATION                      - Analysis
GRADUATION YEAR               - 2014


In [38]:
import pickle
train_data = pickle.load(open('ner_custom_pickle.pkl','rb'))
train_data

DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): MultiHeadSelfAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
    

In [ ]:
nlp_model = spacy.load('nlp_ner_model')

In [41]:
from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer
filepath='my_ner_model/checkpoint-639'

# Named entity recognition pipeline, passing in a specific model and tokenizer
#model = AutoModelForTokenClassification.from_pretrained("dbmdz/bert-large-cased-finetuned-conll03-english")
model=AutoModelForTokenClassification.from_pretrained(filepath)
tokenizer = AutoTokenizer.from_pretrained(filepath)
recognizer = pipeline("ner", model=model, tokenizer=tokenizer)

In [42]:
recognizer

In [52]:
input="Hello, my name is Joe and I live in Los Angeles"
result=recognizer(input)
for x in range(len(result)):
    one_row=result[x]
    entity=input[one_row['start']:one_row['end']]
    entity_name=one_row['entity']
    print(f"NER = {entity} and types= {entity_name}")
    

NER = Joe and types= B-person
NER = Los and types= B-location
NER = Angeles and types= B-location
